In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from skmultilearn.adapt import MLkNN
from scipy.sparse import csr_matrix
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import GridSearchCV
import re
from nltk.stem import WordNetLemmatizer

#Carga de datos
train_data = pd.read_csv("../../Data/train_indexado.csv")
test_data = pd.read_csv("../../Data/test_indexado.csv")

# Definir las clases de emociones
emotion_classes = train_data.columns[2:].tolist()

# --- PREPROCESAMIENTO PARA LEMATIZACIÓN ---
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    words = re.findall(r'\b\w+\b', text.lower())
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized_words)

X_train_lem = train_data['Text'].apply(preprocess_text)
X_test_lem = test_data['Text'].apply(preprocess_text)

# TF-IDF VECTORIZACIÓN
vectorizer = TfidfVectorizer(lowercase=True, strip_accents="unicode", max_features=10000)
X_train = vectorizer.fit_transform(X_train_lem)
X_test = vectorizer.transform(X_test_lem)
y_train = np.asarray(train_data[emotion_classes])
y_test = np.asarray(test_data[emotion_classes])

# Selección de features con Chi2 = 1000
selector = SelectKBest(score_func=chi2, k=1000)
X_train_chi = selector.fit_transform(X_train, y_train)
X_test_chi = selector.transform(X_test)

print(f"Datos preparados: {X_train_chi.shape[0]} muestras de entrenamiento, {X_train_chi.shape[1]} features")

In [ ]:
# Hiperparametrización de MLkNN con Chi2=1000
print("Iniciando búsqueda de hiperparámetros para MLkNN (Chi2=1000)...")

# Parámetros a probar
param_grid = {
    'k': [3, 5, 7, 10],
    's': [0.5, 1.0, 1.5, 2.0]
}

# GridSearchCV manual para MLkNN (no compatible con sklearn GridSearchCV)
best_score = 0
best_params = {}
results = []

for k in param_grid['k']:
    for s in param_grid['s']:
        print(f"Probando k={k}, s={s}")
        
        # Entrenar modelo
        mlknn = MLkNN(k=k, s=s)
        mlknn.fit(X_train_chi, csr_matrix(y_train))
        
        # Predecir
        y_pred = mlknn.predict(X_test_chi)
        
        # Calcular métricas
        report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
        f1_macro = report_dict["macro avg"]["f1-score"]
        accuracy = accuracy_score(y_test, y_pred)
        recall_macro = report_dict["macro avg"]["recall"]
        
        results.append({
            'k': k, 's': s, 
            'accuracy': accuracy, 
            'f1_macro': f1_macro, 
            'recall_macro': recall_macro
        })
        
        print(f"  Accuracy: {accuracy:.5f}, F1: {f1_macro:.5f}, Recall: {recall_macro:.5f}")
        
        # Actualizar mejor resultado
        if f1_macro > best_score:
            best_score = f1_macro
            best_params = {'k': k, 's': s}

print(f"\nMejores parámetros: {best_params}")
print(f"Mejor score (F1 macro): {best_score:.5f}")

In [ ]:
# Evaluar el mejor modelo
best_mlknn = MLkNN(k=best_params['k'], s=best_params['s'])
best_mlknn.fit(X_train_chi, csr_matrix(y_train))
y_pred_best = best_mlknn.predict(X_test_chi)

# Métricas del mejor modelo
report_dict_best = classification_report(y_test, y_pred_best, output_dict=True, zero_division=0)
f1_macro_best = report_dict_best["macro avg"]["f1-score"]
recall_macro_best = report_dict_best["macro avg"]["recall"]

print("\n=== RESULTADOS DEL MEJOR MLkNN (Chi2=1000) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_best):.5f}")
print(f"F1 Score (macro avg): {f1_macro_best:.5f}")
print(f"Recall Score (macro avg): {recall_macro_best:.5f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred_best, zero_division=0)}")

# Comparar con MLkNN por defecto (Chi2=1000)
mlknn_default = MLkNN(k=3)
mlknn_default.fit(X_train_chi, csr_matrix(y_train))
y_pred_default = mlknn_default.predict(X_test_chi)

report_dict_default = classification_report(y_test, y_pred_default, output_dict=True, zero_division=0)
f1_macro_default = report_dict_default["macro avg"]["f1-score"]
recall_macro_default = report_dict_default["macro avg"]["recall"]

print("\n=== COMPARACIÓN CON MLkNN POR DEFECTO (Chi2=1000) ===")
print(f"Accuracy por defecto: {accuracy_score(y_test, y_pred_default):.5f}")
print(f"F1 Score por defecto (macro avg): {f1_macro_default:.5f}")
print(f"Recall Score por defecto (macro avg): {recall_macro_default:.5f}")

print("\n=== MEJORA OBTENIDA ===")
print(f"Mejora en Accuracy: {accuracy_score(y_test, y_pred_best) - accuracy_score(y_test, y_pred_default):.5f}")
print(f"Mejora en F1 Score: {f1_macro_best - f1_macro_default:.5f}")
print(f"Mejora en Recall: {recall_macro_best - recall_macro_default:.5f}")

# Mostrar configuración óptima y tabla de resultados
print("\n=== CONFIGURACIÓN ÓPTIMA ===")
print(f"Chi2 features: 1000")
print(f"k: {best_params['k']}")
print(f"s: {best_params['s']}")

print("\n=== TABLA DE TODOS LOS RESULTADOS ===")
import pandas as pd
df_results = pd.DataFrame(results)
print(df_results.round(5))